# 02 · Metrics and anomaly queries

Known-good SQL for the Genie Agent (example SQL + metric view). Run after `01`.

**Duplicate definition:** a later claim for the same `pet_id + clinic_id + visit_date`. Earliest `submitted_date` is the original. Invoice number is **not** reliable.

**Trap:** two pets from the same household on the same day are **not** duplicates.


In [ ]:
%run ./00_setup


### Q1 — suspected duplicates


In [ ]:
%sql
CREATE OR REPLACE VIEW v_suspected_duplicate_claims AS
WITH ranked AS (
  SELECT c.*,
    ROW_NUMBER() OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS visit_rank,
    FIRST_VALUE(claim_id) OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS original_claim_id,
    FIRST_VALUE(claim_status) OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS original_status
  FROM claims c
)
SELECT * FROM ranked WHERE visit_rank > 1


### Q2 — leakage vs caught


In [ ]:
%sql
SELECT
  COUNT(*) AS suspected_duplicates,
  COUNT_IF(d.claim_status = 'Paid') AS duplicates_paid,
  COUNT_IF(d.claim_status = 'Denied' AND d.denial_reason = 'Duplicate claim') AS duplicates_caught,
  ROUND(SUM(p.payment_amount), 2) AS leakage_usd
FROM v_suspected_duplicate_claims d
LEFT JOIN claim_payments p USING (claim_id)


### Q3 — clinics with the most double submissions


In [ ]:
%sql
SELECT v.clinic_name, COUNT(*) AS suspected_duplicates
FROM v_suspected_duplicate_claims d
JOIN vet_clinics v USING (clinic_id)
GROUP BY v.clinic_name
ORDER BY suspected_duplicates DESC
LIMIT 10


### Trap — household PH0001 (sibling pets must not count as duplicates)


In [ ]:
%sql
SELECT c.claim_id, c.pet_id, p.policyholder_id, c.clinic_id, c.visit_date, c.claim_status
FROM claims c
JOIN pets p USING (pet_id)
WHERE p.policyholder_id = 'PH0001'
ORDER BY c.visit_date, c.pet_id


### Q4 — loss ratio by plan tier and year (stretch)


In [ ]:
%sql
WITH loss AS (
  SELECT pol.plan_tier, YEAR(cp.payment_date) AS yr, SUM(cp.payment_amount) AS paid_losses
  FROM claim_payments cp
  JOIN claims c USING (claim_id)
  JOIN policies pol USING (policy_id)
  GROUP BY ALL
),
prem AS (
  SELECT plan_tier, YEAR(month) AS yr, SUM(earned_premium) AS earned
  FROM earned_premium_monthly
  GROUP BY ALL
)
SELECT plan_tier, yr, paid_losses, earned, ROUND(paid_losses / earned, 3) AS loss_ratio
FROM prem LEFT JOIN loss USING (plan_tier, yr)
ORDER BY plan_tier, yr


### Governed metric view (shared by Genie, SQL, dashboards)


In [ ]:
%sql
CREATE OR REPLACE VIEW claims_metrics
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
source: claims
joins:
  - name: policies
    source: policies
    on: source.policy_id = policies.policy_id
dimensions:
  - name: submitted_month
    expr: DATE_TRUNC('MONTH', submitted_date)
  - expr: diagnosis_category
  - expr: claim_status
  - expr: submission_channel
  - name: plan_tier
    expr: policies.plan_tier
measures:
  - name: claim_count
    expr: COUNT(DISTINCT claim_id)
  - name: approval_rate
    expr: COUNT_IF(claim_status IN ('Approved','Paid')) / NULLIF(COUNT_IF(claim_status IN ('Approved','Paid','Denied')), 0)
    format:
      type: percentage
  - name: denial_rate
    expr: COUNT_IF(claim_status = 'Denied') / NULLIF(COUNT_IF(claim_status IN ('Approved','Paid','Denied')), 0)
    format:
      type: percentage
  - name: avg_days_to_decision
    expr: AVG(DATEDIFF(decision_date, submitted_date))
    synonyms: [processing time, time to settle]
  - name: open_claims
    expr: COUNT_IF(claim_status IN ('Submitted','In Review'))
  - name: total_claimed
    expr: SUM(claimed_amount)
    format:
      type: currency
      currency_code: USD
$$


In [ ]:
%sql
SELECT diagnosis_category, MEASURE(approval_rate), MEASURE(avg_days_to_decision)
FROM claims_metrics
GROUP BY ALL
ORDER BY 1
